# OpenAlex from R: green and circular-economy science in five blocks

**DAISY 2026 summer school, Taranto. Session "Publications as innovation data: measuring (green) science with OpenAlex, at scale with BigQuery"** (Massimiliano Coda Zabetta)

Questions about the circular-economy literature asked to the OpenAlex REST API from R with the `openalexR` package (Aria, Le, Cuccurullo, Belfiore and Choe 2024, *R Journal* 15(4)). A Python twin with `pyalex` (`daisy_openalex_api_python.ipynb`) has the same blocks.

Blocks: **B0** anatomy of a record and cost, **B1** trend, **B2** geography and RTA, **B3** SDG landscape and priorities.

Each block has: **concept** (what and why), **code** (commented, with the Stata equivalent where one exists), **interpret** (what the output says), **exercise** (one line to try after the session).

Runtime: Colab, menu Runtime > Change runtime type > R. Cells run top to bottom; nothing is loaded from disk.

## Setup: packages and API key

Since February 2026 the OpenAlex API expects a key for anything beyond light demo use. The key is free: log in at [openalex.org/settings/api](https://openalex.org/settings/api), copy it.

- With a key: 1 USD of usage per day. A list, filter or group_by call costs 0.0001 USD, a text search 0.001 USD, a single-record lookup nothing.
- Without a key: 0.10 USD per day, about 1,000 calls, enough to run this notebook once.
- Every response carries `meta.cost_usd`, so the price of a call is always visible.

This notebook makes about 320 calls, roughly 0.03 USD; almost all of them are the two uncapped institution downloads of B2 (every page of 200 groups is one call). Keyless still fits one full run.

`openalexR` 3.x follows the current OpenAlex (Walden) schema; version 2.0.0 fails when it parses works. `install.packages()` gets the current CRAN version (3.1.0).

In [ ]:
# Install openalexR from CRAN (about 20 seconds in Colab; needs >= 3.0.0 for the current OpenAlex schema)
install.packages("openalexR", quiet = TRUE)

# dplyr, tidyr and ggplot2 come preinstalled in Colab's R runtime; install them only if missing (local machines)
for (p in c("dplyr", "tidyr", "ggplot2")) if (!requireNamespace(p, quietly = TRUE)) install.packages(p, quiet = TRUE)

In [ ]:
library(openalexR)   # talks to the OpenAlex API
library(dplyr)       # filter, select, mutate, join, summarise (Stata: keep, gen, merge, collapse)
library(tidyr)       # unnest list-columns (Stata: reshape long)
library(ggplot2)     # charts

packageVersion("openalexR")   # should print 3.1.0 or later

In [ ]:
# Your OpenAlex key (free at openalex.org/settings/api). Paste it between the quotes;
# left empty, the notebook runs keyless on the small daily budget, enough for one full run.
YOUR_API_KEY <- ""

options(openalexR.apikey = YOUR_API_KEY)        # register it for the session
options(openalexR.mailto = "you@example.org")   # courtesy contact address (OpenAlex "polite pool")

## B0. Anatomy of one record

**Concept.** One OpenAlex work is a nested JSON record, and everything a publication dataset can measure is a field of it.

Our example is the standard reference of the circular-economy literature: Kirchherr, Reike and Hekkert (2017, *Resources, Conservation and Recycling*), "Conceptualizing the circular economy: an analysis of 114 definitions", OpenAlex id W2756283300, found with a free lookup by DOI (10.1016/j.resconrec.2017.09.005).

The cells below open the record one group at a time: identity and venue, access and impact, the people, the classifications, the text, the links.

`oa_fetch()` turns the record into one tibble row with list-columns; `unnest()` opens them (Stata: `reshape long`).

In [ ]:
# Fetch one work by its OpenAlex id. Single lookups are free; a DOI works too:
# identifier = "https://doi.org/10.1016/j.resconrec.2017.09.005"
paper <- oa_fetch(entity = "works", identifier = "W2756283300")

# One row, 44 columns: the whole record. The cells below open it group by group.
names(paper)

In [ ]:
# Identity and venue: what the paper is and where it appeared.
# glimpse() prints one line per field (Stata: describe plus list in 1)
paper %>%
  select(display_name, publication_year, publication_date, type, language, doi,
         source_display_name, volume, issue) %>%
  glimpse()

In [ ]:
# Access and impact: open access status, total citations, and the FWCI
# (FWCI = 1 means cited exactly as much as the average paper of the same field and year)
paper %>% select(is_oa, oa_status, cited_by_count, fwci) %>% glimpse()

# Citations year by year: a nested table.
# Note the handful dated before publication: online-first copies, dating noise.
paper$counts_by_year[[1]]

In [ ]:
# Authorships: one row per author, with a nested affiliations table.
# unnest() opens it to get institution and country (Stata: reshape long twice, authors then affiliations)
paper$authorships[[1]] %>%
  select(display_name, author_position, affiliations) %>%
  unnest(affiliations, names_sep = "_") %>%
  select(display_name, author_position, affiliations_display_name, affiliations_country_code)

In [ ]:
# Topics: a long table, four rows per topic (type = topic, subfield, field, domain); i = 1 is the primary topic
paper$topics[[1]]

In [ ]:
# SDG tags: NA here, the classifier kept no goal for this paper, abstract and all (keep this in mind for B3)
paper$sustainable_development_goals

# Keywords: algorithmic, with a score (not author keywords)
head(paper$keywords[[1]], 5)

In [ ]:
# The abstract travels as an inverted index (word -> positions, a legal constraint); openalexR rebuilds the text
substr(paper$abstract, 1, 300)

In [ ]:
# The reference list: the works this paper cites, as OpenAlex ids.
# This is the raw material of citation analysis: snowball searches start from it (oa_snowball),
# and patent-to-paper links work the same way.
paper$referenced_works_count          # 95 references (87 resolve to OpenAlex works)
head(paper$referenced_works[[1]], 5)  # the first five ids

In [ ]:
# The price of a call: count_only = TRUE returns only the meta block (count and cost), no records
meta <- oa_fetch(entity = "works", primary_topic.id = "T10471", count_only = TRUE)

meta$count      # works whose primary topic is T10471, Climate Change Policy and Economics (about 105,000)
meta$cost_usd   # 0.0001 USD for a filter call

**Interpret.**

- Three authors, one institution, one country: counting is easy here. On papers with several institutions and countries per author, counting becomes a decision, whole or fractional (B2).
- The primary topic is T10539 "Sustainable Supply Chain Management": no OpenAlex topic is named circular economy, which is why B1 builds a set of topics.
- FWCI 436: cited about 436 times the average paper of its field and year. The citation curve is still rising nine years on, and it shows one or two citations dated before publication: online-first copies and dating noise, a first taste of imperfect linkage.
- The SDG list is empty even though the abstract is stored. SDG tags come from a text classifier (an mBERT model built by the Aurora university network) that keeps only goals predicted above a 0.4 threshold, and a paper about definitions does not read like applied SDG-12 text to the model. Classifier fields are estimates, not ground truth (B3).

**Exercise.** Replace the identifier with the DOI of your own paper (or with W1732240353, Ghisellini, Cialani and Ulgiati 2016, the other standard CE review: no abstract stored, and no SDG tag either) and check its topics and SDGs.

## B1. Trend: is climate and circular-economy science growing faster than science?

**Concept.** The question is a share: of everything published in a year, how much is circular-economy science? A share needs two counts per year:

- the **denominator**: all works OpenAlex indexes that year, with no topic filter at all;
- the **numerator**: the same call with one filter added (`primary_topic.id`), so only works filed under our topics are counted.

`group_by = "publication_year"` does the counting on OpenAlex's servers (Stata: `collapse (count), by(year)`, run remotely): 16 rows come back, not 15 million records.

Topic T10471 is "Climate Change Policy and Economics" (about 105,000 works). For circular economy there is no single topic: we search the topic catalogue for the phrase and use the set of topics the search returns. In the URL the set reads `primary_topic.id:T10539|T11091|...`: values joined by `|` mean OR, and one filter takes up to 100 of them.

In [ ]:
# The denominator: ALL works OpenAlex indexes, counted by year.
# No filter besides the years: articles, books, datasets, everything in the core corpus.
all_year <- oa_fetch(entity = "works", publication_year = "2010-2025", group_by = "publication_year")

# What we downloaded: 16 rows, one per year. key = the year, count = how many works.
# The rows arrive sorted by count, not by year; the merge below reorders them.
all_year

In [ ]:
# A first numerator: the same call, with one filter added
cc_year <- oa_fetch(entity = "works", primary_topic.id = "T10471", publication_year = "2010-2025",
                    group_by = "publication_year")

# 4,000 to 5,000 works per year against 10+ million: shares will be small numbers
head(cc_year, 3)

In [ ]:
# Which topics are about the circular economy? Search the topic catalogue.
# The search reads each topic's name, description and keywords; it returns 8 topics.
ce_topics_tbl <- oa_fetch(entity = "topics", search = "circular economy")
ce_topics_tbl %>% select(display_name, id, works_count)

# The works filter wants bare ids in one OR list ("T10539|T11091|..."):
# strip the URL part, then glue the ids with |
ce_topics <- sub(".*/", "", ce_topics_tbl$id)
ce_or <- paste(ce_topics, collapse = "|")

# The second numerator: CE works by year, the same group_by with the OR list as the filter
ce_year <- oa_fetch(entity = "works", primary_topic.id = ce_or, publication_year = "2010-2025",
                    group_by = "publication_year")

In [ ]:
# One small table per series; as.integer() because the API sends the year as text
den    <- data.frame(year = as.integer(all_year$key), all      = all_year$count)
num_ce <- data.frame(year = as.integer(ce_year$key),  circular = ce_year$count)
num_cc <- data.frame(year = as.integer(cc_year$key),  climate  = cc_year$count)

# Merge them 1:1 on year and sort (Stata: merge 1:1 year, twice, then sort)
trend <- den %>% inner_join(num_ce, by = "year") %>% inner_join(num_cc, by = "year") %>% arrange(year)

# The two shares, in percent (Stata: gen)
trend$circular_share <- 100 * trend$circular / trend$all
trend$climate_share  <- 100 * trend$climate  / trend$all

# One row per year; the two _share columns are what the charts draw
trend

In [ ]:
# The charts stop in 2024: the 2025 total is still being ingested (see the table), so its share is not comparable
trend24 <- subset(trend, year <= 2024)

# One chart per series, same command with a different y column; separate charts because the two levels
# differ by a factor of four to five, so on one axis the climate line would sit flat at the bottom
ggplot(trend24, aes(x = year, y = circular_share)) +
  geom_line() +
  labs(title = "Circular-economy share of world output, 2010-2024", x = NULL, y = "% of all works")

ggplot(trend24, aes(x = year, y = climate_share)) +
  geom_line() +
  labs(title = "Climate-economics share of world output (T10471), 2010-2024", x = NULL, y = "% of all works")

**Interpret.**

- The circular-economy share of world output nearly doubles: 0.13 percent in 2010 to 0.22 in 2024. The set grows faster than science as a whole.
- The climate-economics share (second chart) does not: it slips from 0.053 to 0.036 percent by 2020 and recovers to 0.046 in 2024. One topic is a narrow net: climate economics also lives in energy, environmental science and policy topics.
- The charts stop in 2024. In the table, the 2025 denominator jumps to 14.8 million works (10.7 in 2024) because ingestion of the year is still under way, so the 2025 shares dip for data reasons, not real ones.
- The level depends on the net. A work has one primary topic; the denominator is the whole core corpus (`corpus=core`, the API default, excludes about 190 million low-metadata "xpac" records on both sides); and the 8 search topics are a narrow, defensible core: an earlier 28-topic keyword rule gave shares three times larger. The definition of the field is a choice; one visible search keeps it honest and repeatable.

**Exercise.** Restrict numerator and denominator to `type = "article"` and see whether the shares change.

## B2. Geography: who specialises in circular-economy science?

**Concept.** Three steps:

1. **count works by country**: `group_by = "authorships.countries"` gives, for each country, the works with at least one author affiliated there (whole counting: a paper with Italian and German authors counts once for each);
2. **put the two counts side by side**: CE works and all works, per country;
3. **divide the shares**: a country's share of world CE output over its share of world output is the **RTA** (revealed technological advantage, the Balassa index of the patent literature). Above 1: the country publishes more CE science than its size predicts.

The same logic then runs at institution level, and Italian institutions aggregate to regions.

In [ ]:
# Step 1: works counted by country of the authors' institutions (whole counting), 2020-2025.
# One call for circular-economy works, one for all works.
ce_country  <- oa_fetch(entity = "works", primary_topic.id = ce_or, publication_year = "2020-2025",
                        group_by = "authorships.countries")
all_country <- oa_fetch(entity = "works", publication_year = "2020-2025",
                        group_by = "authorships.countries")

# What we downloaded: one row per country, largest first. key is a URL like https://openalex.org/countries/IT
head(ce_country, 5)
# head(all_country, 5)   # uncomment to inspect the denominator too

In [ ]:
# Step 2: one clean table per series, with the country code cut out of the key URL
ce_c <- data.frame(country      = sub(".*/", "", ce_country$key),
                   country_name = ce_country$key_display_name,
                   ce           = ce_country$count)

all_c <- data.frame(country = sub(".*/", "", all_country$key),
                    all     = all_country$count)

# Merge them 1:1 on country (Stata: merge 1:1 country)
rta_country <- inner_join(ce_c, all_c, by = "country")

# The raw counts: who produces CE science, and how big each country is overall
head(arrange(rta_country, desc(ce)), 10)

In [ ]:
# Step 3: shares, then their ratio
rta_country$ce_share  <- 100 * rta_country$ce  / sum(rta_country$ce)    # share of world CE output
rta_country$all_share <- 100 * rta_country$all / sum(rta_country$all)   # share of world output
rta_country$rta <- rta_country$ce_share / rta_country$all_share         # the RTA: above 1 = specialised

# The same 10 countries, now with shares and RTA next to the raw counts
rta_country <- arrange(rta_country, desc(ce))
head(rta_country, 10)

In [ ]:
# Bar chart: the 20 largest CE producers, ordered by RTA; the dashed line (RTA = 1) separates specialised from not
top20 <- slice_max(rta_country, ce, n = 20)

ggplot(top20, aes(x = reorder(country_name, rta), y = rta)) +
  geom_col() +
  geom_hline(yintercept = 1, linetype = "dashed") +
  coord_flip() +
  labs(title = "RTA in circular-economy science, 2020-2025 (top 20 producers)", x = NULL, y = "RTA")

### Italian regions

Institutions, not countries, carry the fine-grained location: an institution record has a `geo` block with city and region. Three steps:

1. count works by institution: CE works and all works, both restricted to papers with at least one Italian institution;
2. attach each institution's region from its `geo` block;
3. sum by region and compute the same RTA as before, with Italy itself as the reference (RTA = 1).

Step 1 downloads a lot. A `group_by` answer arrives in pages of 200 groups, and openalexR walks through **all** the pages: about 2,400 institution groups for the CE query and 57,000+ for the all-works one (foreign co-author institutions included), about three minutes in total. We accept the wait and get complete counts; `verbose = TRUE` shows the progress while it runs.

In [ ]:
# All institution groups, no cap: the same two group_by calls as before, and openalexR pages through everything.
# The CE query has about 2,400 groups (about a minute); the all-works query 57,000+ (about three minutes).
ce_inst <- oa_fetch(entity = "works", authorships.institutions.country_code = "IT",
                    publication_year = "2020-2025", primary_topic.id = ce_or,
                    group_by = "authorships.institutions.id", verbose = TRUE)

all_inst <- oa_fetch(entity = "works", authorships.institutions.country_code = "IT",
                     publication_year = "2020-2025",
                     group_by = "authorships.institutions.id", verbose = TRUE)

# What we downloaded, largest first (some totals look inflated: institution disambiguation noise)
head(arrange(all_inst, desc(count)), 5)

In [ ]:
# The 200 largest Italian institutions with their geo block (city, region), in one page
inst_it <- oa_fetch(entity = "institutions", country_code = "IT",
                    options = oa_options(sort = "works_count:desc", per_page = 200, pages = 1))

inst_geo <- inst_it %>%
  select(id, display_name, geo) %>%
  unnest(geo, names_sep = "_") %>%
  select(id, display_name, geo_city, geo_region)

# The data problem, before any fix: institutions with a city but no region (81 of the 200)
sum(is.na(inst_geo$geo_region))
inst_geo %>% filter(is.na(geo_region)) %>% head(10)

In [ ]:
# The fix: a city-to-region lookup covering the cities of these 200 institutions (OpenAlex spelling of the regions)
city_region <- c(
  "Ancona" = "The Marches", "Aviano" = "Friuli Venezia Giulia", "Bari" = "Apulia", "Benevento" = "Campania",
  "Bergamo" = "Lombardy", "Bologna" = "Emilia-Romagna", "Bolzano" = "Trentino-Alto Adige", "Brescia" = "Lombardy",
  "Cagliari" = "Sardinia", "Camerino" = "The Marches", "Campobasso" = "Molise", "Candiolo" = "Piedmont",
  "Caserta" = "Campania", "Cassino" = "Lazio", "Catania" = "Sicily", "Catanzaro" = "Calabria",
  "Chieti" = "Abruzzo", "Enna" = "Sicily", "Ferrara" = "Emilia-Romagna", "Fisciano" = "Campania",
  "Florence" = "Tuscany", "Foggia" = "Apulia", "Frascati" = "Lazio", "Genoa" = "Liguria",
  "Ispra" = "Lombardy", "Lavagna" = "Liguria", "Lecce" = "Apulia", "Legnaro" = "Veneto",
  "L’Aquila" = "Abruzzo", "L'Aquila" = "Abruzzo", "Macerata" = "The Marches", "Meldola" = "Emilia-Romagna",
  "Messina" = "Sicily", "Milan" = "Lombardy", "Modena" = "Emilia-Romagna", "Monserrato" = "Sardinia",
  "Monza" = "Lombardy", "Naples" = "Campania", "Padua" = "Veneto", "Palermo" = "Sicily",
  "Parma" = "Emilia-Romagna", "Pavia" = "Lombardy", "Perugia" = "Umbria", "Pino Torinese" = "Piedmont",
  "Pisa" = "Tuscany", "Potenza" = "Basilicata", "Pozzilli" = "Molise", "Reggio Calabria" = "Calabria",
  "Reggio Emilia" = "Emilia-Romagna", "Rende" = "Calabria", "Rome" = "Lazio", "Rozzano" = "Lombardy",
  "San Donato Milanese" = "Lombardy", "San Giovanni Rotondo" = "Apulia", "San Michele all'Adige" = "Trentino-Alto Adige", "Sassari" = "Sardinia",
  "Sesto Fiorentino" = "Tuscany", "Siena" = "Tuscany", "Teramo" = "Abruzzo", "Trento" = "Trentino-Alto Adige",
  "Trieste" = "Friuli Venezia Giulia", "Turin" = "Piedmont", "Udine" = "Friuli Venezia Giulia", "Urbino" = "The Marches",
  "Varese" = "Lombardy", "Venice" = "Veneto", "Vercelli" = "Piedmont", "Verona" = "Veneto",
  "Viterbo" = "Lazio")

# Keep the region when present, else look it up from the city (Stata: replace region = lookup if missing(region))
inst_geo <- inst_geo %>% mutate(region = coalesce(geo_region, unname(city_region[geo_city])))

# Institutions still without a region would be dropped by the joins below; expect 0
sum(is.na(inst_geo$region))

In [ ]:
# Attach each institution's region to its counts and sum by region
# (Stata: merge m:1 institution using geo, then collapse (sum) count, by(region))
ce_region <- ce_inst %>%
  inner_join(inst_geo, by = c("key" = "id")) %>%
  filter(!is.na(region)) %>%
  group_by(region) %>%
  summarise(ce = sum(count), .groups = "drop")

all_region <- all_inst %>%
  inner_join(inst_geo, by = c("key" = "id")) %>%
  filter(!is.na(region)) %>%
  group_by(region) %>%
  summarise(all = sum(count), .groups = "drop")

# The regional counts so far
head(arrange(ce_region, desc(ce)), 5)

In [ ]:
# Same RTA formula as for countries, with Italy itself as the reference (RTA = 1)
rta_region <- left_join(all_region, ce_region, by = "region")
rta_region$ce <- replace_na(rta_region$ce, 0L)

rta_region$rta <- (rta_region$ce / sum(rta_region$ce)) / (rta_region$all / sum(rta_region$all))
rta_region <- arrange(rta_region, desc(rta))

rta_region

In [ ]:
# Bar chart of the regional RTA, regions sorted by RTA; the dashed line (RTA = 1) is the Italian average
ggplot(rta_region, aes(x = reorder(region, rta), y = rta)) +
  geom_col() +
  geom_hline(yintercept = 1, linetype = "dashed") +
  coord_flip() +
  labs(title = "RTA in circular-economy science by Italian region, 2020-2025", x = NULL, y = "RTA")

**Interpret.**

- Raw counts (second table): China leads with about 19,400 CE works in 2020-2025, the United States is a distant second (8,500), Russia third (7,200); Italy is ninth with 3,400.
- The RTA (third table and chart) reorders them: Russia (2.8) is by far the most specialised large producer, because the set's industrial-waste, metallurgy and mineral-processing topics live in Russian applied journals. China (1.3), Brazil (1.2) and India (1.1) publish more than their size predicts; Italy (1.09) sits just above the world average, highest among the large western European producers; the United States (0.45) far below.
- The topic set decides the answer: the earlier 28-topic keyword-rule build had Indonesia fourth (RTA 1.25) and Russia nowhere. Whatever set you use, state it next to the results.
- Regional RTAs rest on institution counts: they double count multi-institution papers, they inherit institution disambiguation errors, and they needed the city patch. Read them as illustrative.
- With the patch, southern and central regions lead: Basilicata 2.4, the Marches 1.9, Campania 1.8, Calabria 1.8, Abruzzo 1.8; Apulia, the region hosting this school, sits at 1.7, well above the Italian average. Lombardy (0.56), Liguria (0.52) and Molise (0.52) close the ranking; the Aosta Valley has no institution among the 200 largest and does not appear.

**Exercise.** Swap the topic set for `"T10471"` and redo the country RTA for climate economics.

## B3. SDG landscape: what are a country's research priorities?

**Concept.** OpenAlex tags works with the 17 UN Sustainable Development Goals (`sustainable_development_goals`; climate action is `https://openalex.org/sdgs/13`). Two questions, roughly one call each:

- a country's **SDG profile**: `group_by = "sustainable_development_goals.id"` for one country (the OpenAlex recipe "Map SDG research");
- its **revealed priorities**: the country's profile divided by the world's, goal by goal. This is the specialisation index of Confraria, Ciarli and Noyons (2024, *Research Policy* 53(3), doi:10.1016/j.respol.2023.104950), who read exactly such indices as countries' research priorities; they compute it on Web of Science with keyword-built SDG queries, we compute it on OpenAlex's classifier tags.

The partner country is China: the largest CE producer of B2, with a profile that differs from Italy's exactly where this school lives (energy, water, environment).

Caveat, planted in B0: SDG tags are classifier output, and databases and classifiers disagree on which papers count (Kashnitsky et al. 2024, *Quantitative Science Studies* 5(2), doi:10.1162/qss_a_00304; Ottaviani and Stahlschmidt 2024, arXiv:2405.03007).

In [ ]:
# The SDG profile of a country: its works 2020-2025, counted by SDG tag (17 rows per country)
sdg_it <- oa_fetch(entity = "works", authorships.countries = "IT", publication_year = "2020-2025",
                   group_by = "sustainable_development_goals.id")
sdg_cn <- oa_fetch(entity = "works", authorships.countries = "CN", publication_year = "2020-2025",
                   group_by = "sustainable_development_goals.id")

# What we downloaded
head(sdg_it, 5)
# head(sdg_cn, 5)   # uncomment to inspect China's table too

In [ ]:
# One clean table per country: SDG label from the key URL (".../sdgs/13" -> "SDG 13"), goal name, count
it <- data.frame(sdg  = paste0("SDG ", sub(".*/", "", sdg_it$key)),
                 goal = sdg_it$key_display_name,
                 n_it = sdg_it$count)
cn <- data.frame(sdg  = paste0("SDG ", sub(".*/", "", sdg_cn$key)),
                 n_cn = sdg_cn$count)

# Merge 1:1 on the goal
sdg <- inner_join(it, cn, by = "sdg")

# Shares of national output: what fraction of a country's works carries each tag.
# The totals come from B2's country table (all_c).
tot_it <- all_c$all[all_c$country == "IT"]
tot_cn <- all_c$all[all_c$country == "CN"]
sdg$share_it <- 100 * sdg$n_it / tot_it
sdg$share_cn <- 100 * sdg$n_cn / tot_cn

arrange(sdg, desc(share_it))

In [ ]:
# Long format for the chart: one row per country and goal (Stata: reshape long)
sdg_long <- sdg %>%
  select(sdg, Italy = share_it, China = share_cn) %>%
  pivot_longer(-sdg, names_to = "country", values_to = "share_pct")

# One bar per goal and country; horizontal so all 17 goal labels are readable, SDG 1 on top
sdg_long$sdg <- factor(sdg_long$sdg, levels = rev(paste0("SDG ", 1:17)))

ggplot(sdg_long, aes(x = sdg, y = share_pct, fill = country)) +
  geom_col(position = "dodge") +
  coord_flip() +
  labs(title = "SDG profile of national output, 2020-2025", x = NULL, y = "% of the country's works", fill = NULL)

In [ ]:
# Revealed research priorities (Confraria, Ciarli and Noyons 2024, eq. 1): for each goal,
# the country's share of its SDG-tagged output over the world's share of world SDG-tagged output.
# Above 1: the country leans on that goal more than world science does.

# The world's SDG profile: the same call, with no country filter
sdg_world <- oa_fetch(entity = "works", publication_year = "2020-2025",
                      group_by = "sustainable_development_goals.id")
world <- data.frame(sdg     = paste0("SDG ", sub(".*/", "", sdg_world$key)),
                    n_world = sdg_world$count)

# Merge and divide
sdg <- inner_join(sdg, world, by = "sdg")
sdg$priority_it <- (sdg$n_it / sum(sdg$n_it)) / (sdg$n_world / sum(sdg$n_world))
sdg$priority_cn <- (sdg$n_cn / sum(sdg$n_cn)) / (sdg$n_world / sum(sdg$n_world))

# Priorities, Italy's highest first
sdg %>% select(sdg, goal, priority_it, priority_cn) %>% arrange(desc(priority_it))

**Interpret.**

- Health dominates both profiles: SDG 3 tags 16.7 percent of Italian works and 12.2 of Chinese. The green goals split the two countries: China puts 10.2 percent of its output on affordable and clean energy (SDG 7) against Italy's 4.2, and 4.4 percent on clean water and sanitation (SDG 6) against Italy's 1.6. One sentence for the room: a coal-heavy energy mix and large state decarbonisation and pollution programmes pull Chinese research toward energy and environmental engineering; Italian output tilts biomedical.
- The priority table reads the same contrast against world science instead of country size: values above 1 mark the goals a country leans on more than the world does. Italy's top revealed priorities are health (1.6), **responsible consumption and production (SDG 12, 1.55)**, the circular-economy goal, an echo of this whole session, and life below water (1.4); China's are affordable and clean energy (2.2) and clean water (2.2). This is the measure of Confraria, Ciarli and Noyons (2024) in miniature: same index, different database and SDG classifier, so levels differ from their paper even where the story agrees.
- All of it is a classifier's reading: repeat the profile with the Scopus or Web of Science SDG filters and the rankings move. That is not a footnote, it is the B0 lesson at country scale.

**Exercise.** Replace China with your own country (or with Indonesia, whose profile is the most different from Italy's in the whole database, largely an artefact of locally indexed education journals) and find its top revealed priority.